У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [2]:
# Import libraries
%run ../Libraries_Imports.ipynb

✓ numpy уже встановлений
✓ pandas уже встановлений
✓ matplotlib уже встановлений
Встановлюю scikit-learn...
✓ statsmodels уже встановлений
✓ sympy уже встановлений
✅ Усі бібліотеки успішно завантажені!


In [3]:
customer_segmentation_df = pd.read_csv('../../Sample Data/customer_segmentation_train.csv')
display(customer_segmentation_df)

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
...,...,...,...,...,...,...,...,...,...,...,...
8063,464018,Male,No,22,No,NaN,0.0,Low,7.0,Cat_1,D
8064,464685,Male,No,35,No,Executive,3.0,Low,4.0,Cat_4,D
8065,465406,Female,No,33,Yes,Healthcare,1.0,Low,1.0,Cat_6,D
8066,467299,Female,No,27,Yes,Healthcare,1.0,Low,4.0,Cat_6,B


In [4]:
customer_segmentation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [5]:
customer_segmentation_df.describe()

,ID,Age,Work_Experience,Family_Size
count,8068.000000,8068.000000,7239.000000,7733.000000
mean,463479.214551,43.466906,2.641663,2.850123
std,2595.381232,16.711696,3.406763,1.531413
min,458982.000000,18.000000,0.000000,1.000000
25%,461240.750000,30.000000,0.000000,2.000000
50%,463472.500000,40.000000,1.000000,3.000000
75%,465744.250000,53.000000,4.000000,4.000000
max,467974.000000,89.000000,14.000000,9.000000


In [6]:
# Прибираємо технічну колонку ID
customer_segmentation_df = customer_segmentation_df.drop(columns=['ID'])

In [7]:
# Обробка пропусків
cat_cols = customer_segmentation_df.select_dtypes(include=['object', 'category']).columns.drop('Segmentation')
num_cols = customer_segmentation_df.select_dtypes(include='number').columns

for col in cat_cols:
    customer_segmentation_df[col] = customer_segmentation_df[col].fillna(customer_segmentation_df[col].mode().iloc[0])

for col in num_cols:
    customer_segmentation_df[col] = customer_segmentation_df[col].fillna(customer_segmentation_df[col].median())

print('Missing values after imputation:')
print(customer_segmentation_df.isna().sum().sort_values(ascending=False))

Missing values after imputation:
Gender             0
Ever_Married       0
Age                0
Graduated          0
Profession         0
Work_Experience    0
Spending_Score     0
Family_Size        0
Var_1              0
Segmentation       0
dtype: int64


In [8]:
# 3) Бінарне кодування ознак з 2 категоріями
binary_maps = {
    'Ever_Married': {'Yes': 1, 'No': 0},
    'Graduated': {'Yes': 1, 'No': 0},
    'Gender': {'Male': 1, 'Female': 0}
}

for col, mapping in binary_maps.items():
    customer_segmentation_df[col] = customer_segmentation_df[col].map(mapping).astype(int)

print(customer_segmentation_df[['Ever_Married', 'Graduated', 'Gender']].head())

   Ever_Married  Graduated  Gender
0             0          0       1
1             1          1       0
2             1          1       0
3             1          1       1
4             1          1       0


In [9]:
# 4) Формуємо X та y
inputs = customer_segmentation_df.drop(columns=['Segmentation'])
targets = customer_segmentation_df['Segmentation']

In [10]:
# 5) Розбиття на train/test (20% test) зі стратифікацією
X_train, X_test, y_train, y_test = train_test_split(
    inputs,
    targets,
    test_size=0.2,
    random_state=12,
    stratify=targets,
)

print('Train shape:', X_train.shape, y_train.shape)
print('Test shape:', X_test.shape, y_test.shape)

Train shape: (6454, 9) (6454,)
Test shape: (1614, 9) (1614,)


In [11]:
# One-Hot кодування
from sklearn.preprocessing import OneHotEncoder

nominal_cols = ['Profession', 'Var_1']

enc = OneHotEncoder(handle_unknown='ignore')
enc.fit(X_train[nominal_cols])

encoded_train = pd.DataFrame(
    enc.transform(X_train[nominal_cols]).toarray(),
    columns=enc.get_feature_names_out(nominal_cols),
    index=X_train.index,
)
encoded_test = pd.DataFrame(
    enc.transform(X_test[nominal_cols]).toarray(),
    columns=enc.get_feature_names_out(nominal_cols),
    index=X_test.index,
)

X_train = pd.concat([X_train.drop(columns=nominal_cols), encoded_train], axis=1)
X_test = pd.concat([X_test.drop(columns=nominal_cols), encoded_test], axis=1)

In [12]:
display(X_train)

,Gender,Ever_Married,Age,Graduated,Work_Experience,Spending_Score,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
4963,1,0,25,0,2.0,Low,7.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6336,0,1,87,0,0.0,Low,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5590,1,1,36,1,7.0,Average,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2743,1,0,48,1,1.0,Low,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5422,1,1,55,1,0.0,Average,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1565,0,0,22,0,0.0,Low,4.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2958,1,1,46,1,7.0,Average,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
7932,0,0,31,1,1.0,Low,3.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5680,0,0,33,0,1.0,Low,8.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [13]:
# Ordinal кодування впорядкованої ознаки Spending_Score
from sklearn.preprocessing import OrdinalEncoder

ordenc = OrdinalEncoder(categories=[['Low', 'Average', 'High']])
ordenc.fit(X_train[['Spending_Score']])

X_train['Spending_Score_Codes'] = ordenc.transform(X_train[['Spending_Score']])
X_test['Spending_Score_Codes'] = ordenc.transform(X_test[['Spending_Score']])

# Після кодування текстовий стовпець більше не потрібен
X_train = X_train.drop(columns=['Spending_Score'])
X_test = X_test.drop(columns=['Spending_Score'])

In [14]:
display(X_train.head())

,Gender,Ever_Married,Age,Graduated,Work_Experience,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,Profession_Entertainment,Profession_Executive,Profession_Healthcare,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,Spending_Score_Codes
4963,1,0,25,0,2.0,7.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6336,0,1,87,0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5590,1,1,36,1,7.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2743,1,0,48,1,1.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5422,1,1,55,1,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [32]:
from imblearn.over_sampling import SMOTE
import pandas as pd

# Набір 1: базовий SMOTE лише на НЕкатегоріальних (числових) ознаках
categorical_like_cols = [
    'Ever_Married',
    'Graduated',
    'Gender',
    'Spending_Score_Codes',
    *[c for c in X_train.columns if c.startswith('Profession_') or c.startswith('Var_1_')],
]
numeric_only_cols = [c for c in X_train.columns if c not in categorical_like_cols]

X_train_numeric = X_train[numeric_only_cols].copy()
X_test_numeric = X_test[numeric_only_cols].copy()

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_numeric, y_train)

print(numeric_only_cols)
print('After SMOTE (numeric-only features):')
print(pd.Series(y_train_smote).value_counts())
print('Shapes:', X_train_smote.shape, y_train_smote.shape)

['Age', 'Work_Experience', 'Family_Size']
After SMOTE (numeric-only features):
Segmentation
D    1814
A    1814
C    1814
B    1814
Name: count, dtype: int64
Shapes: (7256, 3) (7256,)


In [33]:
from imblearn.combine import SMOTETomek

# Набір 2: SMOTE-Tomek
smotetomek = SMOTETomek(random_state=42)
X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train, y_train)

print('After SMOTE-Tomek:')
print(pd.Series(y_train_smotetomek).value_counts())
print('Shapes:', X_train_smotetomek.shape, y_train_smotetomek.shape)

After SMOTE-Tomek:
Segmentation
C    1643
B    1634
D    1619
A    1588
Name: count, dtype: int64
Shapes: (6484, 23) (6484,)


In [34]:
from imblearn.over_sampling import SMOTENC

# SMOTENC
categorical_cols_for_smotenc = [
    'Ever_Married',
    'Graduated',
    'Gender',
    *[c for c in X_train.columns if c.startswith('Profession_') or c.startswith('Var_1_')],
]

categorical_feature_indices = [X_train.columns.get_loc(c) for c in categorical_cols_for_smotenc]

smotenc = SMOTENC(
    categorical_features=categorical_feature_indices,
    random_state=42,
    k_neighbors=5,
 )
X_train_smotenc, y_train_smotenc = smotenc.fit_resample(X_train, y_train)

print('After SMOTENC:')
print(pd.Series(y_train_smotenc).value_counts())
print('Shapes:', X_train_smotenc.shape, y_train_smotenc.shape)

After SMOTENC:
Segmentation
D    1814
A    1814
C    1814
B    1814
Name: count, dtype: int64
Shapes: (7256, 23) (7256,)


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Три моделі OvR Logistic Regression на різних train-наборах
model_sets = [
    ('Original train', X_train, y_train, X_test),
    ('SMOTE train (numeric-only)', X_train_smote, y_train_smote, X_test_numeric),
    ('SMOTE-Tomek train', X_train_smotetomek, y_train_smotetomek, X_test),
]

reports = {}
summary_rows = []

for model_name, X_tr, y_tr, X_te in model_sets:
    clf = LogisticRegression(
        multi_class='ovr',
        solver='lbfgs',
        max_iter=3000,
        random_state=42,
    )
    clf.fit(X_tr, y_tr)

    y_pred = clf.predict(X_te)
    report_dict = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    reports[model_name] = report_dict

    summary_rows.append({
        'Model': model_name,
        'Accuracy': report_dict['accuracy'],
        'Macro F1': report_dict['macro avg']['f1-score'],
        'Weighted F1': report_dict['weighted avg']['f1-score'],
    })

    print('=' * 80)
    print(f'{model_name} - classification_report')
    print(classification_report(y_test, y_pred, zero_division=0))

# Порівняння моделей
summary_df = pd.DataFrame(summary_rows).sort_values(by='Macro F1', ascending=False).reset_index(drop=True)
display(summary_df)

# Обрана метрика для порівняння
chosen_metric = 'Macro F1'
best_model = summary_df.loc[0, 'Model']
best_score = summary_df.loc[0, chosen_metric]
second_score = summary_df.loc[1, chosen_metric]
delta = best_score - second_score

print('Обрана метрика для порівняння:', chosen_metric)
print('Найкраща модель за обраною метрикою:', best_model)
print(f'Найкраще значення {chosen_metric}: {best_score:.4f}')
print(f'Різниця з другим місцем: {delta:.4f}')

Original train - classification_report
              precision    recall  f1-score   support

           A       0.39      0.45      0.42       394
           B       0.38      0.11      0.17       372
           C       0.46      0.69      0.55       394
           D       0.65      0.67      0.66       454

    accuracy                           0.49      1614
   macro avg       0.47      0.48      0.45      1614
weighted avg       0.48      0.49      0.46      1614

SMOTE train (numeric-only) - classification_report
              precision    recall  f1-score   support

           A       0.37      0.34      0.35       394
           B       0.31      0.13      0.18       372
           C       0.41      0.43      0.42       394
           D       0.51      0.75      0.60       454

    accuracy                           0.43      1614
   macro avg       0.40      0.41      0.39      1614
weighted avg       0.40      0.43      0.40      1614

SMOTE-Tomek train - classification_repor

,Model,Accuracy,Macro F1,Weighted F1
0,SMOTE-Tomek train,0.494424,0.467188,0.477317
1,Original train,0.491945,0.451446,0.463015
2,SMOTE train (numeric-only),0.429988,0.389392,0.400270


Обрана метрика для порівняння: Macro F1
Найкраща модель за обраною метрикою: SMOTE-Tomek train
Найкраще значення Macro F1: 0.4672
Різниця з другим місцем: 0.0157


Висновок: різниця помітна; стратегія ресемплінгу впливає на якість моделі.